# Practical 2: Transfer Learning (Cats vs Dogs - MobileNetV2)

## Objective
Use Transfer Learning with MobileNetV2 to classify images of cats and dogs.

## What is Transfer Learning?
Transfer Learning is a technique where we use a pre-trained model (trained on a large dataset like ImageNet) and adapt it for our specific task. This approach:
- Saves training time
- Requires less data
- Often achieves better performance

## MobileNetV2
MobileNetV2 is a lightweight CNN architecture designed for mobile and embedded devices, pre-trained on ImageNet (1.4M images, 1000 classes).

## Step 1: Import Required Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import os

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")

## Step 2: Download and Prepare the Dataset

We'll use TensorFlow's built-in dataset utilities to download the Cats vs Dogs dataset.

In [ ]:
# Download the dataset
dataset_url = "https://storage.googleapis.com/mledu-datasets/cats_and_dogs_filtered.zip"
path_to_zip = keras.utils.get_file('cats_and_dogs.zip', origin=dataset_url, extract=True)
PATH = os.path.join(os.path.dirname(path_to_zip), 'cats_and_dogs_filtered')

train_dir = os.path.join(PATH, 'train')
validation_dir = os.path.join(PATH, 'validation')

print(f"Training directory: {train_dir}")
print(f"Validation directory: {validation_dir}")

# Count images
train_cats_dir = os.path.join(train_dir, 'cats')
train_dogs_dir = os.path.join(train_dir, 'dogs')
validation_cats_dir = os.path.join(validation_dir, 'cats')
validation_dogs_dir = os.path.join(validation_dir, 'dogs')

num_cats_tr = len(os.listdir(train_cats_dir))
num_dogs_tr = len(os.listdir(train_dogs_dir))
num_cats_val = len(os.listdir(validation_cats_dir))
num_dogs_val = len(os.listdir(validation_dogs_dir))

total_train = num_cats_tr + num_dogs_tr
total_val = num_cats_val + num_dogs_val

print(f"\nTotal training cat images: {num_cats_tr}")
print(f"Total training dog images: {num_dogs_tr}")
print(f"Total training images: {total_train}")
print(f"\nTotal validation cat images: {num_cats_val}")
print(f"Total validation dog images: {num_dogs_val}")
print(f"Total validation images: {total_val}")

## Step 3: Data Preprocessing and Augmentation

In [ ]:
# Image dimensions (MobileNetV2 default input size)
IMG_HEIGHT = 224
IMG_WIDTH = 224
BATCH_SIZE = 32

# Training data augmentation
train_image_generator = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Validation data - only rescaling
validation_image_generator = ImageDataGenerator(rescale=1./255)

# Create data generators
train_data_gen = train_image_generator.flow_from_directory(
    batch_size=BATCH_SIZE,
    directory=train_dir,
    shuffle=True,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode='binary'
)

val_data_gen = validation_image_generator.flow_from_directory(
    batch_size=BATCH_SIZE,
    directory=validation_dir,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode='binary'
)

## Step 4: Visualize Sample Images

In [ ]:
# Get a batch of training images
sample_training_images, sample_labels = next(train_data_gen)

# Display the first 5 images
plt.figure(figsize=(15, 5))
for i in range(5):
    plt.subplot(1, 5, i + 1)
    plt.imshow(sample_training_images[i])
    label = 'Dog' if sample_labels[i] == 1 else 'Cat'
    plt.title(label)
    plt.axis('off')
plt.tight_layout()
plt.show()

## Step 5: Load Pre-trained MobileNetV2 Model

In [ ]:
# Load MobileNetV2 pre-trained on ImageNet
# We exclude the top layer (classification layer) as we'll add our own
IMG_SHAPE = (IMG_HEIGHT, IMG_WIDTH, 3)

base_model = MobileNetV2(
    input_shape=IMG_SHAPE,
    include_top=False,
    weights='imagenet'
)

# Freeze the base model layers (don't train them initially)
base_model.trainable = False

print("MobileNetV2 base model loaded successfully!")
print(f"Number of layers in base model: {len(base_model.layers)}")

## Step 6: Build the Transfer Learning Model

In [ ]:
def create_transfer_learning_model(base_model):
    """
    Create a transfer learning model using MobileNetV2 as base
    """
    model = keras.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(1, activation='sigmoid')  # Binary classification
    ])
    
    return model

# Create the model
model = create_transfer_learning_model(base_model)

# Display model architecture
model.summary()

## Step 7: Compile the Model

In [ ]:
# Compile the model
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("Model compiled successfully!")

## Step 8: Train the Model (Initial Training)

In [ ]:
# Define callbacks
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-7
)

# Initial training with frozen base model
EPOCHS = 20

history = model.fit(
    train_data_gen,
    steps_per_epoch=total_train // BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=val_data_gen,
    validation_steps=total_val // BATCH_SIZE,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

## Step 9: Visualize Training History (Initial Training)

In [ ]:
# Plot training history
plt.figure(figsize=(12, 4))

# Accuracy
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy (Initial Training)')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# Loss
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss (Initial Training)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

## Step 10: Fine-tuning (Unfreeze Some Layers)

For better performance, we'll unfreeze the last few layers of MobileNetV2 and train them with a lower learning rate.

In [ ]:
# Unfreeze the base model
base_model.trainable = True

# Fine-tune from this layer onwards
fine_tune_at = 100

# Freeze all layers before the `fine_tune_at` layer
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

print(f"Number of trainable layers: {len([l for l in base_model.layers if l.trainable])}")

## Step 11: Recompile and Fine-tune

In [ ]:
# Recompile with lower learning rate for fine-tuning
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Continue training (fine-tuning)
FINE_TUNE_EPOCHS = 10
TOTAL_EPOCHS = EPOCHS + FINE_TUNE_EPOCHS

history_fine = model.fit(
    train_data_gen,
    steps_per_epoch=total_train // BATCH_SIZE,
    epochs=TOTAL_EPOCHS,
    initial_epoch=len(history.history['loss']),
    validation_data=val_data_gen,
    validation_steps=total_val // BATCH_SIZE,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

## Step 12: Visualize Complete Training History

In [ ]:
# Combine histories
acc = history.history['accuracy'] + history_fine.history['accuracy']
val_acc = history.history['val_accuracy'] + history_fine.history['val_accuracy']
loss = history.history['loss'] + history_fine.history['loss']
val_loss = history.history['val_loss'] + history_fine.history['val_loss']

# Plot complete training history
plt.figure(figsize=(12, 4))

# Accuracy
plt.subplot(1, 2, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.axvline(x=len(history.history['accuracy'])-1, color='r', linestyle='--', label='Start Fine-tuning')
plt.title('Model Accuracy (Complete Training)')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# Loss
plt.subplot(1, 2, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.axvline(x=len(history.history['loss'])-1, color='r', linestyle='--', label='Start Fine-tuning')
plt.title('Model Loss (Complete Training)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

## Step 13: Evaluate on Validation Set

In [ ]:
# Evaluate on validation data
val_loss, val_accuracy = model.evaluate(val_data_gen, steps=total_val // BATCH_SIZE, verbose=0)

print(f"\nValidation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_accuracy:.4f} ({val_accuracy*100:.2f}%)")

## Step 14: Make Predictions and Visualize Results

In [ ]:
# Reset validation generator
val_data_gen.reset()

# Get predictions for validation set
predictions = model.predict(val_data_gen, steps=total_val // BATCH_SIZE)
predicted_classes = (predictions > 0.5).astype(int).flatten()

# Get true labels
val_data_gen.reset()
true_classes = val_data_gen.classes[:len(predicted_classes)]

# Display some predictions
val_data_gen.reset()
sample_images, sample_labels = next(val_data_gen)
sample_predictions = model.predict(sample_images)
sample_pred_classes = (sample_predictions > 0.5).astype(int).flatten()

plt.figure(figsize=(15, 8))
for i in range(min(20, len(sample_images))):
    plt.subplot(4, 5, i + 1)
    plt.imshow(sample_images[i])
    true_label = 'Dog' if sample_labels[i] == 1 else 'Cat'
    pred_label = 'Dog' if sample_pred_classes[i] == 1 else 'Cat'
    confidence = sample_predictions[i][0] if sample_pred_classes[i] == 1 else 1 - sample_predictions[i][0]
    color = 'green' if sample_labels[i] == sample_pred_classes[i] else 'red'
    plt.title(f"True: {true_label}\nPred: {pred_label} ({confidence:.2f})", color=color, fontsize=9)
    plt.axis('off')
plt.tight_layout()
plt.show()

## Step 15: Confusion Matrix

In [ ]:
# Create confusion matrix
cm = confusion_matrix(true_classes, predicted_classes)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Cat', 'Dog'], yticklabels=['Cat', 'Dog'])
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

## Step 16: Classification Report

In [ ]:
# Print classification report
print("\nClassification Report:")
print("=" * 60)
print(classification_report(true_classes, predicted_classes, target_names=['Cat', 'Dog']))

## Step 17: Save the Model (Optional)

In [ ]:
# Save the model
# model.save('cats_vs_dogs_mobilenetv2.h5')
# print("Model saved successfully!")

## Summary

### What we learned:
1. **Transfer Learning**: Using a pre-trained model (MobileNetV2) for a new task
2. **Data Augmentation**: Artificially expanding the dataset to improve generalization
3. **Two-stage Training**:
   - **Stage 1**: Train only the custom classifier layers with frozen base model
   - **Stage 2**: Fine-tune the last few layers of the base model with lower learning rate
4. **Binary Classification**: Cat vs Dog classification with sigmoid activation
5. **Model Evaluation**: Accuracy, confusion matrix, and classification report

### Key Concepts:
- **Transfer Learning**: Leveraging knowledge from pre-trained models
- **Fine-tuning**: Unfreezing and training some layers of the pre-trained model
- **MobileNetV2**: Efficient CNN architecture for mobile devices
- **Data Augmentation**: Random transformations to increase training data variety
- **GlobalAveragePooling2D**: Reduces spatial dimensions while preserving channels
- **Binary Crossentropy**: Loss function for binary classification
- **Sigmoid Activation**: Outputs probability for binary classification

### Advantages of Transfer Learning:
1. **Less Training Data Required**: Pre-trained features work well on new tasks
2. **Faster Training**: Only train a few layers instead of the entire network
3. **Better Performance**: Benefits from features learned on large datasets (ImageNet)
4. **Reduced Computational Cost**: Smaller models train faster